# 19. Language Models

attention의 weighted value, causal mask, 다음 token 예측을 직접 계산합니다.

핵심 실습은 NumPy와 Python 표준 라이브러리만 사용합니다. API 호출, 모델 다운로드, GPU는 사용하지 않습니다. 수치는 고정된 작은 입력의 계산이며 실제 pretrained model의 성능이 아닙니다.

In [1]:
import numpy as np
np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(27)

## Attention의 한 행

고정 logits와 values에서 temperature가 혼합값을 어떻게 바꾸는지 계산합니다.

In [2]:
def softmax(x,axis=-1):
    z=x-np.max(x,axis=axis,keepdims=True)
    z=np.exp(z)
    return z/z.sum(axis=axis,keepdims=True)
logits=np.array([1.,.5,-.5]);values=np.array([2.,4.,8.])
for temperature in [.2,1.,5.]:
    weights=softmax(logits/temperature)
    print(temperature, weights, weights@values)
assert np.isclose(softmax(logits)@values,3.3947128328551925)

0.2 [0.9237 0.0758 0.0005] 2.154704056784603
1.0 [0.5465 0.3315 0.122 ] 3.394707834706556
5.0 [0.378 0.342 0.28 ] 4.364096093471091


## Causal mask

허용하지 않은 위치는 softmax 이전에 `-inf`로 막습니다. 점수를 0으로 바꾸는 것은 같은 연산이 아닙니다.

In [3]:
q=np.array([[1.,0.],[0.,1.],[1.,1.]])
k=q.copy();v=np.array([[1.,0.],[0.,2.],[3.,1.]])
scores=q@k.T/np.sqrt(q.shape[1])
allowed=np.tri(len(q),dtype=bool)
weights=softmax(np.where(allowed,scores,-np.inf))
out=weights@v
print("causal weights:\n",weights)
print("output:\n",out)
assert np.all(weights[np.triu_indices(3,1)]==0)
assert np.allclose(weights.sum(axis=1),1)

causal weights:
 [[1.     0.     0.    ]
 [0.3302 0.6698 0.    ]
 [0.2483 0.2483 0.5035]]
output:
 [[1.     0.    ]
 [0.3302 1.3395]
 [1.7587 1.    ]]


## 다음 토큰의 학습 예

학습된 LLM을 흉내 낸 것이 아니라, 조건부 확률이라는 목적을 작은 bigram count로 분리합니다.

In [4]:
from collections import Counter,defaultdict
corpus=["the cat sat", "the cat slept", "the dog sat"]
counts=defaultdict(Counter)
for sentence in corpus:
    tokens=sentence.split()+["<eos>"]
    for left,right in zip(tokens,tokens[1:]):counts[left][right]+=1
def next_distribution(token):
    row=counts[token];total=sum(row.values())
    return {k:v/total for k,v in row.items()}
print("after the:",next_distribution("the"))
print("after cat:",next_distribution("cat"))
assert next_distribution("the")["cat"]==2/3

after the: {'cat': 0.6666666666666666, 'dog': 0.3333333333333333}
after cat: {'sat': 0.5, 'slept': 0.5}


## 과제

- 첫 query의 출력이 미래 token의 value를 바꿔도 유지되는지 확인하세요. mask를 제거한 경우와 비교하세요.
- 온도를 크게 만들었을 때 weighted value가 values의 단순 평균에 가까워지는지 확인하세요.

In [5]:
# 과제 풀이 1
v_changed=v.copy();v_changed[-1]=[1000.,1000.]
masked_changed=weights@v_changed
assert np.allclose(out[0],masked_changed[0])
unmasked=softmax(scores)@v
unmasked_changed=softmax(scores)@v_changed
assert not np.allclose(unmasked[0],unmasked_changed[0])
# 과제 풀이 2
near_uniform=softmax(logits/1e6)@values
print("large-temperature mixture:",near_uniform,"mean:",values.mean())
assert np.isclose(near_uniform,values.mean(),atol=1e-5)

large-temperature mixture: 4.666665111111296 mean: 4.666666666666667


## 문맥과 weight

위 입력과 mask를 바꾸면 계산 결과는 달라지지만 q·k의 학습 파라미터를 update한 것은 아닙니다. ICL·retrieval은 입력을 바꾸는 경로이고 fine-tuning은 weight를 바꿉니다. [Transformer](https://arxiv.org/abs/1706.03762), [BERT](https://arxiv.org/abs/1810.04805), [Instruction tuning](https://arxiv.org/abs/2203.02155).